In [1]:
import wmfdata as wmf
import pandas as pd

In [153]:
query = wmf.spark.run("""
WITH registered_users AS (
    SELECT
        mwh.event_user_text AS user_name,
        MIN(mwh.event_user_creation_timestamp) AS registration_date
    FROM wmf.mediawiki_history mwh
    INNER JOIN canonical_data.wikis wikis ON mwh.wiki_db = wikis.database_code
    WHERE
        size(mwh.event_user_is_bot_by) = 0
        AND size(mwh.event_user_is_bot_by_historical) = 0
        AND mwh.snapshot = '2024-01'
        AND wikis.database_group IN (
            "commons", "incubator", "foundation", "mediawiki", "meta", "sources",
            "species", "wikibooks", "wikidata", "wikifunctions", "wikinews",
            "wikipedia", "wikiquote", "wikisource", "wikiversity", "wikivoyage",
            "wiktionary"
        )
    GROUP BY mwh.event_user_text
    HAVING registration_date BETWEEN '2024-01-01' AND '2024-02-01'
),
user_content_edits AS (
    SELECT
        mwh.event_user_text AS user_name,
        SUM(
    CASE 
    WHEN COALESCE(page_namespace_is_content_historical, FALSE) THEN 1 
    ELSE 0 
    END
) AS content_edits

    FROM wmf.mediawiki_history mwh
    INNER JOIN canonical_data.wikis wikis ON mwh.wiki_db = wikis.database_code
    INNER JOIN registered_users ru ON mwh.event_user_text = ru.user_name
    WHERE
        mwh.event_entity = 'revision'
        AND mwh.event_type = 'create'
        AND mwh.snapshot = '2024-01'
        AND mwh.event_timestamp BETWEEN '2024-01-01' AND '2024-02-01'
        AND wikis.database_group IN (
            "commons", "incubator", "foundation", "mediawiki", "meta", "sources",
            "species", "wikibooks", "wikidata", "wikifunctions", "wikinews",
            "wikipedia", "wikiquote", "wikisource", "wikiversity", "wikivoyage",
            "wiktionary"
        )
    GROUP BY mwh.event_user_text
    HAVING content_edits >= 5
)
SELECT
    ru.user_name,
    ru.registration_date,
    uce.content_edits
FROM user_content_edits uce
INNER JOIN registered_users ru ON uce.user_name = ru.user_name;


""")

KeyboardInterrupt: 

In [124]:

query.to_csv('df_historical.csv', index = False)

In [38]:
fixed = wmf.spark.run("""WITH earliest_registration AS (
    SELECT
        user_name,
        MIN(user_registration) AS earliest_registration_date
    FROM wmf_product.editor_month
    WHERE
        user_id != 0
    GROUP BY user_name
),
global_edits AS (
    SELECT
        em.month,
        em.user_name,
        SUM(em.content_edits) AS content_edits,
        MAX(em.bot_by_group) AS bot_by_group,
        CAST(TRUNC(er.earliest_registration_date, 'MONTH') AS DATE) AS registration_month
    FROM wmf_product.editor_month em
    INNER JOIN earliest_registration er ON em.user_name = er.user_name
    WHERE
        em.MONTH = '2024-01-01'
        AND em.user_id != 0
        AND em.user_name NOT REGEXP 'bot\\b'
        --and em.user_name = 'Adexgifted'
    GROUP BY em.month, em.user_name, er.earliest_registration_date
)
SELECT
 * 
FROM global_edits
--WHERE content_edits >= 5
--    AND NOT bot_by_group
--GROUP BY month;""")

fixed

,month,user_name,content_edits,bot_by_group,registration_month
0,2024-01-01,.ankitt 09,0,False,2024-01-01
1,2024-01-01,08dle,0,False,2024-01-01
2,2024-01-01,09CI,2,False,2024-01-01
3,2024-01-01,25Punkte,11,False,2019-06-01
4,2024-01-01,56GW56,1,False,2024-01-01
...,...,...,...,...,...
291780,2024-01-01,村上 雅弥,1,False,2022-08-01
291781,2024-01-01,松井信雄,21,False,2017-01-01
291782,2024-01-01,桃花君,1,False,2024-01-01
291783,2024-01-01,楊騏瑋,58,False,2020-03-01


In [6]:
df = pd.read_csv("df.csv")
df_historical = pd.read_csv("df_historical.csv")

#df_historical = query.copy()

In [42]:
#fixed['registration'] = pd.to_datetime(fixed['registration_month'])

filtered_rows = fixed[fixed['registration_month'] == pd.Timestamp('2024-01-01')]

/home/hghani/.conda/envs/2023-08-15T23.19.45_hghani/lib/python3.10/site-packages/pandas/core/ops/array_ops.py:73: FutureWarning: Comparison of Timestamp with datetime.date is deprecated in order to match the standard library behavior. In a future version these will be considered non-comparable. Use 'ts == pd.Timestamp(date)' or 'ts.date() == date' instead.
  result = libops.scalar_compare(x.ravel(), y, op)


In [43]:
filtered_rows=filtered_rows[filtered_rows['content_edits'] > 4]

In [15]:
missing = set(filtered_rows.user_name) - set(df.user_name)


In [127]:
df_historical

,user_name,registration_date,content_edits
0,666Pak,2024-01-13 14:27:18.0,5
1,Abdul Rashid Seikh lp,2024-01-01 10:47:06.0,24
2,Alina Sulowska,2024-01-01 01:18:38.0,31
3,Apocj,2024-01-22 08:11:50.0,68
4,AsjaMas,2024-01-09 17:33:20.0,6
...,...,...,...
15430,Unknown23467,2024-01-14 02:09:18.0,5
15431,User34264n,2024-01-10 10:57:16.0,6
15432,XxItssDanielaxX,2024-01-03 12:29:46.0,17
15433,کاراندیش,2024-01-09 16:44:26.0,8


In [10]:
len(result2.user_name.unique())

16174

In [57]:
result2['min_registration_date'] = pd.to_datetime(result2['min_registration_date'])
filtered_result = result2[result2['min_registration_date'] > '2023-12']


In [58]:
filtered_result

,user_name,total_edit_count,month,min_registration_date
1,666Pak,5,2024-01,2024-01-13 14:27:18
5,Abdul Rashid Seikh lp,23,2024-01,2024-01-01 10:47:06
6,Adamfragz,18,2024-01,2023-12-30 10:56:34
11,Alina Sulowska,31,2024-01,2024-01-01 01:18:38
17,Apocj,21,2024-01,2024-01-22 08:11:50
...,...,...,...,...
83997,Thinkonomics,7,2024-01,2024-01-25 00:54:08
84010,Unknown23467,5,2024-01,2024-01-14 02:09:18
84024,Xoxoxoxoko,64,2024-01,2023-12-16 20:32:50
84025,XxItssDanielaxX,17,2024-01,2024-01-03 12:29:46


In [59]:
len(filtered_result.user_name.unique())

16174

In [16]:
query = wmf.spark.run("""
select month, country_code, namespace_zero_distinct_editors,  activity_level  from wmf.unique_editors_by_country_monthly
where month = '2024-01'
and activity_level != '1 to 4'
and users_are_anonymous = FALSE

""")

query

,month,country_code,namespace_zero_distinct_editors,activity_level
0,2024-01,US,2058,100 or more
1,2024-01,US,11309,5 to 99


In [17]:
query.namespace_zero_distinct_editors.sum()

13367

In [18]:
df

,user_name,registration_date,content_edits
0,666Pak,2024-01-13 14:27:17.0,5
1,Abdul Rashid Seikh lp,2024-01-01 10:47:06.0,24
2,Alina Sulowska,2024-01-01 01:18:37.0,31
3,Apocj,2024-01-22 08:11:49.0,68
4,AsjaMas,2024-01-09 17:33:20.0,6
...,...,...,...
15483,Unknown23467,2024-01-14 02:09:17.0,5
15484,User34264n,2024-01-10 10:57:16.0,6
15485,XxItssDanielaxX,2024-01-03 12:29:46.0,17
15486,کاراندیش,2024-01-09 16:44:25.0,8


In [29]:
fixed[fixed['user_name'] == 'Adexgifted']

,month,user_name,content_edits,bot_by_group,registration_month
0,2024-01-01,Adexgifted,6,False,2024-01-01


In [30]:
df[df['user_name'] == 'Adexgifted']

,user_name,registration_date,content_edits


In [ ]:
fixed[fixed['user_name'] == A

In [132]:
missing_in_df = df_historical[~df_historical['user_name'].isin(filtered_rows.user_name)]

In [133]:
missing_in_df

,user_name,registration_date,content_edits
5995,R i p,2024-01-02 02:26:11.0,11
7665,Tetsugaku-San,2024-01-26 17:05:10.0,9
8016,Leendenvano,2024-01-25 16:41:31.0,5
10162,Zpcola,2024-01-14 23:37:25.0,21
10859,EvanTPeoples,2024-01-11 00:42:37.0,13
11300,Mkellner,2024-01-01 19:02:16.0,5
11502,E2kk6n,2024-01-18 15:27:24.0,16


In [110]:
df_historical[df_historical['user_name'] == 'Sparsumb']
filtered_rows[filtered_rows['user_name'] == 'Sparsumb']

,month,user_name,content_edits,bot_by_group,registration_month


In [130]:
missing_in_fixed = filtered_rows[~filtered_rows['user_name'].isin(df_historical.user_name)]

In [131]:
missing_in_fixed[missing_in_fixed['bot_by_group'] == False]

,month,user_name,content_edits,bot_by_group,registration_month
147,2024-01-01,Bookybo,6,False,2024-01-01
1644,2024-01-01,Karlheinz Deppe-Wiesinger,6,False,2024-01-01
1760,2024-01-01,Mbjpxncp7k,31,False,2024-01-01
4025,2024-01-01,LuciusFlamma,21,False,2024-01-01
4104,2024-01-01,Molinaaraniva,5,False,2024-01-01
...,...,...,...,...,...
287478,2024-01-01,DALILUA,9,False,2024-01-01
287508,2024-01-01,Dillweedbagadae,8,False,2024-01-01
289355,2024-01-01,Valerii Kolpakov,5,False,2024-01-01
290135,2024-01-01,Mliz2000,23,False,2024-01-01


In [119]:
filtered_rows

,month,user_name,content_edits,bot_by_group,registration_month
5,2024-01-01,666Pak,5,False,2024-01-01
18,2024-01-01,Abdul Rashid Seikh lp,24,False,2024-01-01
51,2024-01-01,Alina Sulowska,31,False,2024-01-01
74,2024-01-01,Apocj,68,False,2024-01-01
88,2024-01-01,AsjaMas,6,False,2024-01-01
...,...,...,...,...,...
291657,2024-01-01,Unknown23467,5,False,2024-01-01
291660,2024-01-01,User34264n,6,False,2024-01-01
291707,2024-01-01,XxItssDanielaxX,17,False,2024-01-01
291763,2024-01-01,کاراندیش,8,False,2024-01-01


In [120]:
df_historical

,user_name,registration_date,content_edits
0,666Pak,2024-01-13 14:27:17.0,5
1,Abdul Rashid Seikh lp,2024-01-01 10:47:06.0,24
2,Alina Sulowska,2024-01-01 01:18:37.0,31
3,Apocj,2024-01-22 08:11:49.0,68
4,AsjaMas,2024-01-09 17:33:20.0,6
...,...,...,...
15427,Unknown23467,2024-01-14 02:09:17.0,5
15428,User34264n,2024-01-10 10:57:16.0,6
15429,XxItssDanielaxX,2024-01-03 12:29:46.0,17
15430,کاراندیش,2024-01-09 16:44:25.0,8


In [81]:
filtered_rows

,month,user_name,content_edits,bot_by_group,registration_month
5,2024-01-01,666Pak,5,False,2024-01-01
18,2024-01-01,Abdul Rashid Seikh lp,24,False,2024-01-01
51,2024-01-01,Alina Sulowska,31,False,2024-01-01
74,2024-01-01,Apocj,68,False,2024-01-01
88,2024-01-01,AsjaMas,6,False,2024-01-01
...,...,...,...,...,...
291657,2024-01-01,Unknown23467,5,False,2024-01-01
291660,2024-01-01,User34264n,6,False,2024-01-01
291707,2024-01-01,XxItssDanielaxX,17,False,2024-01-01
291763,2024-01-01,کاراندیش,8,False,2024-01-01


In [4]:
result2 = wmf.spark.run("""

SELECT wiki_db, event_user_text, event_user_creation_timestamp, event_timestamp, event_entity, event_type, page_namespace, page_namespace_historical, page_namespace_is_content_historical, event_user_registration_timestamp, page_namespace_is_content
FROM wmf.mediawiki_history
WHERE event_user_text in ('Serol1971','Valerii Kolpakov' )
and snapshot = '2024-01'
AND NOT (SIZE(event_user_is_bot_by) > 0 OR SIZE(event_user_is_bot_by_historical) > 0)
ORDER BY event_user_creation_timestamp

 

""")

result2

,wiki_db,event_user_text,event_user_creation_timestamp,event_timestamp,event_entity,event_type,page_namespace,page_namespace_historical,page_namespace_is_content_historical,event_user_registration_timestamp,page_namespace_is_content
0,enwiki,Serol1971,2008-06-17 08:36:01.0,2008-06-17 08:36:00.0,user,create,NaN,NaN,None,2008-06-17 08:36:00.0,None
1,enwiki,Serol1971,2008-06-17 08:36:01.0,2012-08-21 13:07:14.0,revision,create,0.0,0.0,True,2008-06-17 08:36:00.0,True
2,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-08-23 13:46:05.0,revision,create,3.0,3.0,False,2008-06-17 08:36:00.0,False
3,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-08-23 13:45:02.0,revision,create,3.0,3.0,False,2008-06-17 08:36:00.0,False
4,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-08-23 13:45:31.0,revision,create,3.0,3.0,False,2008-06-17 08:36:00.0,False
...,...,...,...,...,...,...,...,...,...,...,...
56,wikidatawiki,Valerii Kolpakov,2024-01-25 17:59:56.0,2024-01-26 09:15:54.0,revision,create,0.0,0.0,True,2024-01-25 17:59:53.0,True
57,wikidatawiki,Valerii Kolpakov,2024-01-25 17:59:56.0,2024-01-26 09:22:07.0,revision,create,0.0,0.0,True,2024-01-25 17:59:53.0,True
58,wikidatawiki,Valerii Kolpakov,2024-01-25 17:59:56.0,2024-01-26 09:23:24.0,revision,create,0.0,0.0,True,2024-01-25 17:59:53.0,True
59,wikidatawiki,Valerii Kolpakov,2024-01-25 17:59:56.0,2024-01-26 17:16:23.0,revision,create,0.0,0.0,True,2024-01-25 17:59:53.0,True


In [5]:
filtered_result2 = result2[pd.to_datetime(result2['event_user_registration_timestamp']) < pd.Timestamp('2024-01-01')]

filtered_result2

,wiki_db,event_user_text,event_user_creation_timestamp,event_timestamp,event_entity,event_type,page_namespace,page_namespace_historical,page_namespace_is_content_historical,event_user_registration_timestamp,page_namespace_is_content
0,enwiki,Serol1971,2008-06-17 08:36:01.0,2008-06-17 08:36:00.0,user,create,NaN,NaN,None,2008-06-17 08:36:00.0,None
1,enwiki,Serol1971,2008-06-17 08:36:01.0,2012-08-21 13:07:14.0,revision,create,0.0,0.0,True,2008-06-17 08:36:00.0,True
2,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-08-23 13:46:05.0,revision,create,3.0,3.0,False,2008-06-17 08:36:00.0,False
3,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-08-23 13:45:02.0,revision,create,3.0,3.0,False,2008-06-17 08:36:00.0,False
4,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-08-23 13:45:31.0,revision,create,3.0,3.0,False,2008-06-17 08:36:00.0,False
5,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-08-23 13:44:32.0,revision,create,3.0,3.0,False,2008-06-17 08:36:00.0,False
6,enwiki,Serol1971,2008-06-17 08:36:01.0,2013-04-08 09:55:35.0,revision,create,7.0,7.0,False,2008-06-17 08:36:00.0,False
7,enwiki,Serol1971,2008-06-17 08:36:01.0,2013-04-08 10:07:42.0,revision,create,0.0,0.0,True,2008-06-17 08:36:00.0,True
8,enwiki,Serol1971,2008-06-17 08:36:01.0,2013-04-08 10:09:22.0,revision,create,0.0,0.0,True,2008-06-17 08:36:00.0,True
9,enwiki,Serol1971,2008-06-17 08:36:01.0,2011-03-22 17:17:12.0,revision,create,0.0,0.0,True,2008-06-17 08:36:00.0,True


24/02/29 16:51:33 WARN Client: Exception encountered while connecting to the server : org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
24/02/29 16:51:39 WARN Client: Exception encountered while connecting to the server : org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error


In [91]:
filtered_rows_to_csv("fixed.csv", index = False)

NameError: name 'filtered_rows_to_csv' is not defined

In [137]:
filtered_rows

,month,user_name,content_edits,bot_by_group,registration_month
5,2024-01-01,666Pak,5,False,2024-01-01
18,2024-01-01,Abdul Rashid Seikh lp,24,False,2024-01-01
51,2024-01-01,Alina Sulowska,31,False,2024-01-01
74,2024-01-01,Apocj,68,False,2024-01-01
88,2024-01-01,AsjaMas,6,False,2024-01-01
...,...,...,...,...,...
291657,2024-01-01,Unknown23467,5,False,2024-01-01
291660,2024-01-01,User34264n,6,False,2024-01-01
291707,2024-01-01,XxItssDanielaxX,17,False,2024-01-01
291763,2024-01-01,کاراندیش,8,False,2024-01-01


In [138]:
df_historical

,user_name,registration_date,content_edits
0,666Pak,2024-01-13 14:27:18.0,5
1,Abdul Rashid Seikh lp,2024-01-01 10:47:06.0,24
2,Alina Sulowska,2024-01-01 01:18:38.0,31
3,Apocj,2024-01-22 08:11:50.0,68
4,AsjaMas,2024-01-09 17:33:20.0,6
...,...,...,...
15430,Unknown23467,2024-01-14 02:09:18.0,5
15431,User34264n,2024-01-10 10:57:16.0,6
15432,XxItssDanielaxX,2024-01-03 12:29:46.0,17
15433,کاراندیش,2024-01-09 16:44:26.0,8


In [140]:
list = missing_in_fixed.user_name.unique()
list = "'" + "', '".join(list) + "'"
list

"'Bookybo', 'Karlheinz Deppe-Wiesinger', 'Mbjpxncp7k', 'LuciusFlamma', 'Molinaaraniva', 'Vidyavachaspati Vidyanand', 'Wssbot', 'Miss Universe Editor.Bot', 'آرش حسینی بازیگر', 'Alpha beta boy', 'Panisexualka', 'Evbot99', 'Олександр Патейчук', 'HyperDiary', 'Pelinsu89', 'Raindooo', 'Elia Frankin', 'Ahmed.khalil12410', 'Editsssss phhhhhh bot', 'Pgoering', 'Tq93sgd5', 'Almitu', 'Bercanda Saja', 'Tom Alexander Azevedo', 'IXAWAI', 'InformasiBot', 'Fusionfingers', 'Babylas48', 'Bulalo bot', 'Lg1236', 'CorrectBioBot', 'Gaia1st', 'Flagswodum', 'Indicebot', 'Bot Baku', 'Lukeysu', 'Cocofai', 'Yyahmedsakr', 'Ericnarro', 'Atomicfart2009', 'Fractalize', 'Dickman78', 'Minnerbot XD', 'Mwmaury', 'SamiTK-T2', 'Escarda', 'XPosBot', 'Habib5858', 'Danielandreasbeck', 'Jabbot1794', 'Mubury', 'Italianoa 9000', 'Quintoxey', 'Fofadja', 'DrChabuk', 'UništihEkonomiju', 'Jonafa', 'AlexanderBot04', 'Mxolisi Ncube', 'Threehundredsongs', 'Kissmhie1', 'Poleo Cintia', 'Ludoholidays', 'Bot1984', 'E931u04g4ru,4', 'Ewwbo

In [142]:
result2 = wmf.spark.run(f"""
SELECT wiki_db, 
       event_user_text, 
       event_user_creation_timestamp, 
       event_timestamp, 
       event_entity, 
       event_type, 
       page_namespace, 
       page_namespace_historical, 
       page_namespace_is_content_historical, 
       event_user_registration_timestamp, 
       page_namespace_is_content
FROM wmf.mediawiki_history
WHERE event_user_text IN ({list})
AND snapshot = '2024-01'
AND NOT (SIZE(event_user_is_bot_by) > 0 OR SIZE(event_user_is_bot_by_historical) > 0)
ORDER BY event_user_creation_timestamp


 

""")

result2

,wiki_db,event_user_text,event_user_creation_timestamp,event_timestamp,event_entity,event_type,page_namespace,page_namespace_historical,page_namespace_is_content_historical,event_user_registration_timestamp,page_namespace_is_content
0,zhwiki,Mbjpxncp7k,None,2024-01-03 12:36:44.0,revision,create,118.0,118.0,False,2023-09-24 09:00:58.0,False
1,zhwiki,Mbjpxncp7k,None,2024-01-04 11:32:11.0,revision,create,118.0,118.0,False,2023-09-24 09:00:58.0,False
2,zhwiki,Mbjpxncp7k,None,2024-01-06 14:53:34.0,revision,create,2.0,2.0,False,2023-09-24 09:00:58.0,False
3,zhwiki,Mbjpxncp7k,None,2023-12-02 16:54:51.0,revision,create,0.0,0.0,True,2023-09-24 09:00:58.0,True
4,zhwiki,Mbjpxncp7k,None,2023-12-04 15:01:04.0,revision,create,0.0,0.0,True,2023-09-24 09:00:58.0,True
...,...,...,...,...,...,...,...,...,...,...,...
3509,urwiki,Abu1ameen,2024-01-31 15:31:55.0,2024-01-31 16:11:22.0,page,create,0.0,0.0,True,2024-01-31 15:31:54.0,True
3510,dewiki,Abu1ameen,2024-01-31 15:32:03.0,2024-01-31 15:32:01.0,user,create,NaN,NaN,None,2024-01-31 15:32:01.0,None
3511,hewiki,Abu1ameen,2024-01-31 15:32:04.0,2024-01-31 15:32:01.0,user,create,NaN,NaN,None,2024-01-31 15:32:01.0,None
3512,enwikibooks,Lukeysu,2024-02-01 02:10:08.0,2024-02-01 02:10:08.0,user,create,NaN,NaN,None,2024-02-01 02:10:08.0,None


In [144]:
len(result2.event_user_text.unique())

99

In [148]:
import pandas as pd

from datetime import datetime

df = result2.copy()

# Convert columns to datetime, taking care of None and NaN

df['event_user_creation_timestamp'] = pd.to_datetime(df['event_user_creation_timestamp'], errors='coerce')

df['event_timestamp'] = pd.to_datetime(df['event_timestamp'], errors='coerce')

df['event_user_registration_timestamp'] = pd.to_datetime(df['event_user_registration_timestamp'], errors='coerce')



# Check for dates less than 2024-01-01 in any of the timestamp columns

cutoff_date = datetime(2024, 1, 1)

mask = (df['event_user_creation_timestamp'] < cutoff_date) | (df['event_timestamp'] < cutoff_date) | (df['event_user_registration_timestamp'] < cutoff_date)



# Count unique users matching the condition

unique_users_before_2024 = df[mask]['event_user_text'].nunique()

unique_users_before_2024

99

In [152]:
unique_users_with_none_creation_timestamp = df[df['event_user_creation_timestamp'].isna()]['event_user_text'].unique()


unique_users_with_none_creation_timestamp 

array(['Mbjpxncp7k', 'Ttcath'], dtype=object)

In [156]:
fixed[fixed['user_name'] == 'Valerii Kolpakov']

,month,user_name,content_edits,bot_by_group,registration_month
289355,2024-01-01,Valerii Kolpakov,5,False,2024-01-01


In [7]:
df

,user_name,registration_date,content_edits
0,666Pak,2024-01-13 14:27:17.0,5
1,Abdul Rashid Seikh lp,2024-01-01 10:47:06.0,24
2,Alina Sulowska,2024-01-01 01:18:37.0,31
3,Apocj,2024-01-22 08:11:49.0,68
4,AsjaMas,2024-01-09 17:33:20.0,6
...,...,...,...
15483,Unknown23467,2024-01-14 02:09:17.0,5
15484,User34264n,2024-01-10 10:57:16.0,6
15485,XxItssDanielaxX,2024-01-03 12:29:46.0,17
15486,کاراندیش,2024-01-09 16:44:25.0,8


In [9]:
df_historical.sort_values(by='registration_date', ascending=False)


,user_name,registration_date,content_edits
7549,Shltrainkid,2024-01-31 23:42:18.0,14
3033,LegitBritman,2024-01-31 23:36:32.0,5
14955,KANA-BiSU,2024-01-31 23:33:13.0,7
2877,Perilou,2024-01-31 23:31:54.0,8
5527,TarasBulba88,2024-01-31 23:16:35.0,8
...,...,...,...
12831,顏家良,2024-01-01 00:38:38.0,6
12029,KeyGue,2024-01-01 00:37:23.0,11
5638,Rey161203,2024-01-01 00:31:00.0,55
8302,Seraj salem,2024-01-01 00:21:51.0,38


24/02/29 21:36:50 WARN Client: Exception encountered while connecting to the server : org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
24/02/29 21:36:54 WARN Client: Exception encountered while connecting to the server : org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
24/02/29 21:37:03 WARN Client: Exception encountered while connecting to the server : org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
24/02/29 21:37:42 WARN Client: Exception encountered while connecting to the server : org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state